# BÁO CÁO THU THẬP DỮ LIỆU THỜI TIẾT


## 1. Tổng quan
Thu thập dữ liệu thời tiết lịch sử từ Open-Meteo Historical Weather API, chuyển dữ liệu thành dạng bảng và lưu dưới các định dạng CSV, TSV và JSON.

In [ ]:
import time
import pandas as pd
import requests
from pathlib import Path

## 2. Nguồn dữ liệu<br>
Nguồn: Open-Meteo Historical Weather API<br>
API: https://archive-api.open-meteo.com/v1/archive <br>
Thời gian: từ 01/01/2021 đến 31/12/2025<br>
Tần suất: theo ngày<br>
Địa điểm: Thành phố Hồ Chí Minh, Hà Nội, Đà Nẵng, Đà Lạt và Cần Thơ

In [ ]:
api_url = "https://archive-api.open-meteo.com/v1/archive"
start_date = "2021-01-01"
end_date = "2025-12-31"

In [ ]:
cities = [
    {"name": "Ho Chi Minh City", "latitude": 10.8231, "longitude": 106.6297},
    {"name": "Ha Noi", "latitude": 21.0285, "longitude": 105.8542},
    {"name": "Da Nang", "latitude": 16.0544, "longitude": 108.2022},
    {"name": "Da Lat", "latitude": 11.9404, "longitude": 108.4583},
    {"name": "Can Tho", "latitude": 10.0452, "longitude": 105.7469},
]

In [ ]:
output_directory = Path("/content/data")
dfs = []

## 3. Mô tả các thuộc tính

Mỗi dòng biểu diễn dữ liệu thời tiết của một thành phố trong một ngày.
Các thuộc tính chính:

`city`: tên thành phố.</br>
`time`: ngày quan sát.</br>
`weather_code`: mã trạng thái thời tiết theo chuẩn WMO.</br>
`temperature_2m_max`: nhiệt độ cao nhất trong ngày, đơn vị °C.</br>
`temperature_2m_min`: nhiệt độ thấp nhất trong ngày, đơn vị °C.</br>
`temperature_2m_mean`: nhiệt độ trung bình trong ngày, đơn vị °C.</br>
`precipitation_sum`: tổng lượng mưa trong ngày, đơn vị mm.</br>
`precipitation_hours`: số giờ có mưa trong ngày.</br>
`sunshine_duration`: thời gian có nắng, đơn vị giây.</br>
`wind_speed_10m_max`: tốc độ gió lớn nhất ở độ cao 10 mét, đơn vị km/h.</br>

In [ ]:
for city in cities:
    params = {
        "latitude": city["latitude"],
        "longitude": city["longitude"],
        "start_date": start_date,
        "end_date": end_date,
        "daily":(
            "weather_code",
            "temperature_2m_max",
            "temperature_2m_min",
            "temperature_2m_mean",
            "precipitation_sum",
            "precipitation_hours",
            "sunshine_duration",
            "wind_speed_10m_max",
        ),
        "timezone": "Asia/Ho_Chi_Minh",
    }
    response = requests.get(api_url,params=params)
    result = response.json()
    df = pd.DataFrame(result["daily"])
    df.insert(0,"city",city["name"])
    dfs.append(df)
    time.sleep(1)

In [ ]:
weather_df = pd.concat(dfs, ignore_index=True)

In [ ]:
weather_df.to_csv(output_directory/ "weather_daily.csv", index=False, encoding="utf-8-sig")
weather_df.to_csv(output_directory/ "weather_daily.tsv", index=False, sep="\t", encoding="utf-8-sig")
weather_df.to_json(output_directory/ "weather_daily.json", orient="records", force_ascii=False, indent=2)

## 4. Thống kê cơ bản về bộ dữ liệu


### 1. Kích thước bộ dữ liệu

In [ ]:
print("Số dòng:", weather_df.shape[0])
print("Số cột:", weather_df.shape[1])
print("Số thành phố:", weather_df["city"].nunique())

Số dòng: 9130
Số cột: 10
Số thành phố: 5


### 2. Kiểm tra dữ liệu thiếu và trùng

In [ ]:
print("Giá trị thiếu:", weather_df.isna().sum())
duplicate_count = weather_df.duplicated(subset=["city","time"]).sum()
print("Số dòng trùng:", duplicate_count)

Giá trị thiếu: city                   0
time                   0
weather_code           0
temperature_2m_max     0
temperature_2m_min     0
temperature_2m_mean    0
precipitation_sum      0
precipitation_hours    0
sunshine_duration      0
wind_speed_10m_max     0
dtype: int64
Số dòng trùng: 0


### 3. Thống kê theo thành phố

So sánh nhiệt độ trung bình, nhiệt độ cao nhất, nhiệt độ thấp nhất, tổng lượng mưa và tốc độ gió lớn nhất giữa các thành phố.

In [ ]:
city_summary = (
    weather_df.groupby("city")
    .agg(
        average_temperature=("temperature_2m_mean","mean"),
        highest_temperature=("temperature_2m_max","max"),
        lowest_temperature=("temperature_2m_min","min"),
        total_precipitation=("precipitation_sum","sum"),
        maximum_wind_speed=("wind_speed_10m_max","max"),
    )
)
display(city_summary)

,average_temperature,highest_temperature,lowest_temperature,total_precipitation,maximum_wind_speed
city,,,,,
Can Tho,26.991895,38.3,18.8,11918.6,30.8
Da Lat,18.844140,30.8,8.8,9111.4,39.3
Da Nang,26.237568,40.2,15.1,12235.1,59.2
Ha Noi,24.131106,40.2,6.1,10975.1,62.4
Ho Chi Minh City,27.497919,39.2,17.9,12159.0,30.1
